# Default Model BehaviorBench Evaluation

In [5]:
from pathlib import Path
import sys

import torch

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'src').is_dir():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / 'src'))
sys.path.insert(0, str(REPO_ROOT / 'scripts'))

from eval.behaviorbench import evaluate_behaviorbench
from run_caa import _load_model
from steering.common_config import MODEL_NAME

OUTPUT_PATH = REPO_ROOT / 'artifacts' / 'eval' / 'default_behaviorbench.csv'
DATA_PATH = REPO_ROOT / 'data' / 'behaviorbench' / 'adapted' / 'data' / 'test.json'

In [6]:
def load_default_model():
    """Load the project default model without a steering hook."""
    try:
        model, tokenizer = _load_model()
        runtime = '4-bit NF4'
    except ImportError as error:
        if 'bitsandbytes' not in str(error):
            raise
        from transformers import AutoModelForCausalLM, AutoTokenizer

        device = 'mps' if torch.backends.mps.is_available() else 'cpu'
        dtype = torch.float16 if device == 'mps' else torch.float32
        tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, local_files_only=True)
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            local_files_only=True,
            torch_dtype=dtype,
            low_cpu_mem_usage=True,
        ).to(device)
        runtime = f'unquantized {dtype} fallback'

    tokenizer.padding_side = 'left'
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    model.eval()
    return model, tokenizer, runtime


model, tokenizer, runtime = load_default_model()
print(f'Loaded {MODEL_NAME}')
print(f'Runtime: {runtime}')
print(f'Device: {next(model.parameters()).device}')

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loaded meta-llama/Llama-3.1-8B-Instruct
Runtime: unquantized torch.float16 fallback
Device: mps:0


In [7]:
baseline_rows = evaluate_behaviorbench(
    model=model,
    tokenizer=tokenizer,
    output_csv=OUTPUT_PATH,
    data_path=DATA_PATH,
    method='default',
    split='test',
    max_new_tokens=8,
)

print(f'Wrote {len(baseline_rows)} rows to {OUTPUT_PATH}')

Wrote 25 rows to /Users/arya/Documents/GT/CS7650 - NLP/Project/llm-personality-conditioning/artifacts/eval/default_behaviorbench.csv


In [8]:
from IPython.display import display
import pandas as pd

summary = pd.DataFrame(baseline_rows)
display(summary[['trait', 'target_score', 'endorsement_rate', 'invalid_rate', 'n', 'valid_n']])

,trait,target_score,endorsement_rate,invalid_rate,n,valid_n
0,Openness,1,0.833333,0.0,30,30
1,Openness,2,0.833333,0.0,30,30
2,Openness,3,0.833333,0.0,30,30
3,Openness,4,0.833333,0.0,30,30
4,Openness,5,0.833333,0.0,30,30
5,Conscientiousness,1,0.766667,0.0,30,30
6,Conscientiousness,2,0.766667,0.0,30,30
7,Conscientiousness,3,0.766667,0.0,30,30
8,Conscientiousness,4,0.766667,0.0,30,30
9,Conscientiousness,5,0.766667,0.0,30,30
